In [1]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, AutoConfig
from safetensors.torch import save_file
import json
import os
from huggingface_hub import login, HfApi, create_repo
from google.colab import userdata
import gc

In [13]:
hf_token = userdata.get('HF_TOKEN_WRITE')
login(token=hf_token)

In [3]:
# Load the full model one last time
full_model = AutoModelForCausalLM.from_pretrained(
    "google/medgemma-1.5-4b-it",
    trust_remote_code=True,
    dtype=torch.bfloat16
)

tokenizer = AutoTokenizer.from_pretrained("google/medgemma-1.5-4b-it")

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

In [4]:
# Assuming full_model and tokenizer are already loaded
output_dir = "./medgemma-1.5-4b-it-text-only"
os.makedirs(output_dir, exist_ok=True)

# Shard size (adjust if needed)
max_shard_size = 2 * 1024 * 1024 * 1024  # 2GB

shard_idx = 0
current_shard = {}
current_shard_size = 0
weight_map = {}

# Get all keys first
all_keys = list(full_model.state_dict().keys())

for old_key in all_keys:
    # Skip non-language model weights
    if not (old_key.startswith("model.language_model.") or old_key == "lm_head.weight"):
        continue

    # Determine new key
    if old_key.startswith("model.language_model."):
        new_key = old_key.replace("model.language_model.", "model.")
    else:
        new_key = old_key

    # Load single tensor
    tensor = full_model.state_dict()[old_key]
    tensor_size = tensor.numel() * tensor.element_size()

    # Check if new shard needed
    if current_shard_size + tensor_size > max_shard_size and current_shard:
        shard_filename = f"model-{shard_idx+1:05d}-of-XXXXX.safetensors"
        print(f"Saving {shard_filename} ({current_shard_size / 1024 / 1024:.2f} MB)")

        save_file(current_shard, f"{output_dir}/{shard_filename}")

        # Free memory immediately
        current_shard = {}
        current_shard_size = 0
        gc.collect()

        shard_idx += 1

    # Add to current shard
    if "embed_tokens.weight" in new_key or new_key == "lm_head.weight":
        current_shard[new_key] = tensor.cpu().clone()
    else:
        current_shard[new_key] = tensor.cpu()

    current_shard_size += tensor_size

    # Define shard_filename for weight_map
    shard_filename = f"model-{shard_idx+1:05d}-of-XXXXX.safetensors"
    weight_map[new_key] = shard_filename

# Save last shard
if current_shard:
    shard_filename = f"model-{shard_idx+1:05d}-of-XXXXX.safetensors"
    print(f"Saving {shard_filename} ({current_shard_size / 1024 / 1024:.2f} MB)")
    save_file(current_shard, f"{output_dir}/{shard_filename}")

total_shards = shard_idx + 1

# Rename with correct counts
for i in range(total_shards):
    old_name = f"{output_dir}/model-{i+1:05d}-of-XXXXX.safetensors"
    new_name = f"{output_dir}/model-{i+1:05d}-of-{total_shards:05d}.safetensors"
    os.rename(old_name, new_name)

    # Update weight_map
    old_shard_name = f"model-{i+1:05d}-of-XXXXX.safetensors"
    new_shard_name = f"model-{i+1:05d}-of-{total_shards:05d}.safetensors"
    for key in weight_map:
        if weight_map[key] == old_shard_name:
            weight_map[key] = new_shard_name

# Calculate total size from all final shard files
total_size = 0
for shard_file in set(weight_map.values()):
    shard_path = os.path.join(output_dir, shard_file)
    if os.path.exists(shard_path):
        total_size += os.path.getsize(shard_path)
    else:
        print(f"Warning: Missing shard file {shard_path}")

# Create the complete index structure
index = {
    "metadata": {
        "total_size": total_size
    },
    "weight_map": weight_map
}

# Save the index with metadata
with open(f"{output_dir}/model.safetensors.index.json", "w") as f:
    json.dump(index, f, indent=2)

print(f"\n✅ Model saved in {total_shards} shards!")
print(f"    Total size: {total_size / 1024**3 :.2f} GB")
print(f"    Index file created with metadata (total_size = {total_size:,} bytes)")

Saving model-00001-of-XXXXX.safetensors (2030.40 MB)
Saving model-00002-of-XXXXX.safetensors (2030.23 MB)
Saving model-00003-of-XXXXX.safetensors (2030.23 MB)
Saving model-00004-of-XXXXX.safetensors (1310.17 MB)
Saving model-00005-of-XXXXX.safetensors (1280.31 MB)

✅ Model saved in 5 shards!
    Total size: 8.48 GB
    Index file created with metadata (total_size = 9,103,082,712 bytes)


In [5]:
# Even simpler approach - use the text_config directly
original_config = full_model.config

# Get the text config as a dict
text_config_dict = original_config.text_config.to_dict()

# Create new config based on text config
config_dict = {
    "architectures": ["Gemma3ForCausalLM"],
    "model_type": "gemma3",
    "torch_dtype": "bfloat16",
    "transformers_version": "4.48.0",
    "tie_word_embeddings": False,
}

# Merge with text config
config_dict.update(text_config_dict)

with open(f"{output_dir}/config.json", "w") as f:
    json.dump(config_dict, f, indent=2)

tokenizer.save_pretrained(output_dir)

print("✅ Config and tokenizer saved!")

✅ Config and tokenizer saved!


In [6]:
# Load directly - no need for full model!
model = AutoModelForCausalLM.from_pretrained(
    "./medgemma-1.5-4b-it-text-only",
    dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True
)

tokenizer = AutoTokenizer.from_pretrained("./medgemma-1.5-4b-it-text-only")

Loading weights:   0%|          | 0/445 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


In [7]:
# Test it
prompt = """<bos><start_of_turn>user
What can a headache be?<end_of_turn>
<start_of_turn>model"""

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

with torch.inference_mode():
    outputs = model.generate(
        **inputs,
        max_new_tokens=180,
        do_sample=True,
        temperature=0.75,
        top_p=0.92,
        repetition_penalty=1.1,
    )

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

user
What can a headache be?
model
A headache is a common symptom that can have many different causes. Here's a breakdown of what it could potentially be:

**Common Causes:**

* **Tension Headaches:** These are the most frequent type. They often feel like a constant ache or pressure around the head, especially at the temples or forehead. Stress, fatigue, poor posture, and muscle tension in the neck and shoulders are common triggers.
* **Migraines:** Migraines are more severe headaches, often accompanied by nausea, vomiting, sensitivity to light and sound, and sometimes an aura (visual disturbances). Triggers can include hormonal changes, stress, certain foods, and sensory stimuli.
* **Sinus Headaches:** Caused by inflammation or congestion in the sinuses (air-filled cavities within your skull). Symptoms often include pain and pressure in the forehead, cheeks, or bridge of the nose, along


In [11]:
# Your HuggingFace username and desired repo name
username = "gabrielbuzzi"  # Replace with your actual username
repo_name = "medgemma-1.5-4b-it-text-only"
repo_id = f"{username}/{repo_name}"

In [15]:
# Upload all files from your local directory
api = HfApi()

local_dir = "./medgemma-1.5-4b-it-text-only"

# Upload the entire folder
api.upload_folder(
    folder_path=local_dir,
    repo_id=repo_id,
    repo_type="model",
    commit_message="Upload text-only version of MedGemma 1.5 4B IT",
    token=hf_token
)

print(f"✅ Model uploaded successfully!")
print(f"🔗 View at: https://huggingface.co/{repo_id}")

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...-text-only/tokenizer.json:  75%|#######5  | 25.2MB / 33.4MB            

  ...0003-of-00005.safetensors:   0%|          |  538kB / 2.13GB            

  ...0004-of-00005.safetensors:   1%|1         | 16.7MB / 1.37GB            

  ...0002-of-00005.safetensors:   0%|          | 8.39MB / 2.13GB            

  ...0005-of-00005.safetensors:   1%|1         | 16.8MB / 1.34GB            

  ...0001-of-00005.safetensors:   1%|          | 16.9MB / 2.13GB            

✅ Model uploaded successfully!
🔗 View at: https://huggingface.co/gabrielbuzzi/medgemma-1.5-4b-it-text-only
